# MoodSignal - Stock Market Prediction via Social Sentiment

Scrapes StockTwits + Reddit, scores sentiment (FinBERT/VADER), computes mood dimensions, trains XGBoost to predict next-day DJIA direction.

---

## Step 1: Install Dependencies

In [ ]:
!pip install -q praw "requests==2.32.4" transformers torch vaderSentiment yfinance xgboost scikit-learn pandas numpy python-dotenv tabulate curl_cffi

## Step 2: Configuration

In [ ]:
import os
from pathlib import Path

# -- Paths --
BASE_DIR = Path("/content/MoodSignal")
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RAW_POSTS_CSV = DATA_DIR / "raw_posts.csv"
SCORED_POSTS_CSV = DATA_DIR / "scored_posts.csv"
DAILY_MOOD_CSV = DATA_DIR / "daily_mood.csv"
DJIA_PRICES_CSV = DATA_DIR / "djia_prices.csv"
FEATURES_CSV = DATA_DIR / "features.csv"
MODEL_PATH = MODEL_DIR / "xgb_model.json"

# -- Reddit (optional -- fill in if you have keys) --
REDDIT_CLIENT_ID = ""         # your_client_id
REDDIT_CLIENT_SECRET = ""     # your_client_secret
REDDIT_USER_AGENT = "MoodSignal/1.0"
SUBREDDITS = ["wallstreetbets", "stocks", "investing"]

# -- StockTwits (public, no auth) --
STOCKTWITS_BASE_URL = "https://api.stocktwits.com/api/2"
STOCKTWITS_SYMBOLS = [
    "DJI", "SPY", "AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA",
    "META", "JPM", "BAC", "GS", "V", "BA", "DIS",
]

# -- Market --
DJIA_TICKER = "^DJI"
LOOKBACK_DAYS = 90

# -- Sentiment --
FINBERT_MODEL = "ProsusAI/finbert"
FINBERT_WEIGHT = 0.6
VADER_WEIGHT = 0.4

# -- Model --
TRAIN_TEST_SPLIT = 0.8
XGB_PARAMS = {
    "max_depth": 4,
    "n_estimators": 100,
    "learning_rate": 0.1,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
}

# -- Mood keywords --
FEAR_KEYWORDS = [
    "crash", "dump", "sell", "bear", "recession", "panic", "collapse",
    "plunge", "tank", "fear", "warning", "crisis", "bubble", "risk",
]
HAPPY_KEYWORDS = [
    "moon", "rocket", "bull", "rally", "gain", "profit", "surge",
    "boom", "breakout", "diamond", "buy", "long", "calls", "tendies",
]

USE_FINBERT = False  # Set True if you want FinBERT (slower but better)

print("Config loaded.")

## Step 3: Scrape StockTwits Data

In [ ]:
import time
import requests
import pandas as pd
from datetime import datetime, timezone

try:
    from curl_cffi import requests as requests_cffi
    HAS_CFFI = True
except ImportError:
    HAS_CFFI = False

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36",
    "Accept": "application/json, text/html,*/*",
    "Referer": "https://stocktwits.com/",
}

rows = []
for symbol in STOCKTWITS_SYMBOLS:
    url = f"{STOCKTWITS_BASE_URL}/streams/symbol/{symbol}.json"
    print(f"  Fetching ${symbol}... ", end="")
    try:
        if HAS_CFFI:
            # Use curl_cffi to bypass Cloudflare challenges on Google Colab
            resp = requests_cffi.get(url, impersonate="chrome120", timeout=15)
        else:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            if resp.status_code == 403:
                time.sleep(5)
                resp = requests.get(url, headers=HEADERS, timeout=15)

        resp.raise_for_status()
        messages = resp.json().get("messages", [])
        for msg in messages[:30]:
            body = msg.get("body", "").strip()
            if not body:
                continue
            created = msg.get("created_at", "")
            try:
                ts = datetime.strptime(created, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc).isoformat()
            except:
                ts = datetime.now(timezone.utc).isoformat()
            rows.append({
                "text": body[:500].replace("\n", " "),
                "timestamp": ts,
                "source": "stocktwits",
                "subreddit": symbol,
                "score": msg.get("likes", {}).get("total", 0) if isinstance(msg.get("likes"), dict) else 0,
                "num_comments": 0,
            })
        print(f"{len([r for r in rows if r['subreddit']==symbol])} msgs")
    except Exception as e:
        print(f"ERROR: {e}")
    time.sleep(1.5)

raw_df = pd.DataFrame(rows)
raw_df.drop_duplicates(subset=["text", "timestamp"], inplace=True)
raw_df.to_csv(RAW_POSTS_CSV, index=False)
print(f"\nTotal posts collected: {len(raw_df)}")
raw_df.head()


## Step 3b (Optional): Scrape Reddit with PRAW
Fill in your Reddit API keys in Step 2 first.

In [ ]:
if REDDIT_CLIENT_ID and REDDIT_CLIENT_ID != "your_client_id":
    import praw
    reddit = praw.Reddit(
        client_id=REDDIT_CLIENT_ID,
        client_secret=REDDIT_CLIENT_SECRET,
        user_agent=REDDIT_USER_AGENT,
    )
    reddit_rows = []
    for sub_name in SUBREDDITS:
        print(f"  Scraping r/{sub_name}...")
        subreddit = reddit.subreddit(sub_name)
        for submission in subreddit.hot(limit=50):
            text = submission.title
            if submission.selftext:
                text += f" {submission.selftext[:500]}"
            reddit_rows.append({
                "text": text.replace("\n", " ").strip(),
                "timestamp": datetime.fromtimestamp(submission.created_utc, tz=timezone.utc).isoformat(),
                "source": "reddit",
                "subreddit": sub_name,
                "score": submission.score,
                "num_comments": submission.num_comments,
            })
            submission.comments.replace_more(limit=0)
            for comment in submission.comments[:5]:
                if comment.body and comment.body not in ("[deleted]", "[removed]"):
                    reddit_rows.append({
                        "text": comment.body[:500].replace("\n", " ").strip(),
                        "timestamp": datetime.fromtimestamp(comment.created_utc, tz=timezone.utc).isoformat(),
                        "source": "reddit",
                        "subreddit": sub_name,
                        "score": comment.score,
                        "num_comments": 0,
                    })
    reddit_df = pd.DataFrame(reddit_rows)
    combined = pd.concat([raw_df, reddit_df], ignore_index=True)
    combined.drop_duplicates(subset=["text", "timestamp"], inplace=True)
    combined.to_csv(RAW_POSTS_CSV, index=False)
    print(f"Added {len(reddit_df)} Reddit posts. Total: {len(combined)}")
else:
    print("Reddit keys not set -- skipping. StockTwits data is sufficient.")

## Step 4: Sentiment Analysis (VADER + optional FinBERT)

In [ ]:
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

finbert_pipeline = None
if USE_FINBERT:
    from transformers import pipeline
    print("Loading FinBERT model (first run downloads ~400MB)...")
    finbert_pipeline = pipeline("sentiment-analysis", model=FINBERT_MODEL, tokenizer=FINBERT_MODEL, truncation=True, max_length=512)
    print("FinBERT loaded.")

df = pd.read_csv(RAW_POSTS_CSV)
print(f"Scoring {len(df)} posts...")

results = []
for i, row in df.iterrows():
    text = str(row.get("text", ""))
    v = vader.polarity_scores(text)
    vader_scores = {"vader_compound": v["compound"], "vader_pos": v["pos"], "vader_neg": v["neg"], "vader_neu": v["neu"]}

    if finbert_pipeline:
        try:
            fb = finbert_pipeline(text[:512])[0]
            fb_label, fb_score = fb["label"], fb["score"]
        except:
            fb_label, fb_score = "neutral", 0.0
    else:
        fb_label, fb_score = "neutral", 0.0

    fb_numeric = {"positive": 1.0, "negative": -1.0, "neutral": 0.0}.get(fb_label, 0.0)
    fb_value = fb_numeric * fb_score
    combined = (FINBERT_WEIGHT * fb_value + VADER_WEIGHT * v["compound"]) if USE_FINBERT else v["compound"]

    results.append({**vader_scores, "finbert_label": fb_label, "finbert_score": round(fb_score, 4), "sentiment_score": round(combined, 4)})
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(df)}")

scored_df = pd.concat([df.reset_index(drop=True), pd.DataFrame(results)], axis=1)
scored_df.to_csv(SCORED_POSTS_CSV, index=False)

avg = scored_df["sentiment_score"].mean()
pos_pct = (scored_df["sentiment_score"] > 0.1).mean() * 100
neg_pct = (scored_df["sentiment_score"] < -0.1).mean() * 100
print(f"\nDone! Avg sentiment: {avg:+.3f} | Positive: {pos_pct:.1f}% | Negative: {neg_pct:.1f}%")
scored_df[["text", "sentiment_score", "finbert_label"]].head(10)

## Step 5: Compute Daily Mood Dimensions

In [ ]:
df = pd.read_csv(SCORED_POSTS_CSV)
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
df["date"] = df["timestamp"].dt.date

def keyword_boost(texts, keywords):
    lower = texts.str.lower()
    return lower.apply(lambda t: any(kw in str(t) for kw in keywords)).mean()

daily = []
for date, group in df.groupby("date"):
    if len(group) < 2:
        continue
    scores = group["sentiment_score"].astype(float)
    texts = group["text"].fillna("")
    engagement = np.log1p(group["score"].fillna(0).clip(lower=0) + 1)
    ew = engagement / engagement.sum() if engagement.sum() > 0 else None

    std = scores.std()
    calm = float(np.clip(1.0 - std, 0, 1))
    strongly_neg = (scores < -0.3).mean()
    fear = float(np.clip(0.6 * strongly_neg + 0.4 * keyword_boost(texts, FEAR_KEYWORDS), 0, 1))
    pos_mask = scores > 0
    if pos_mask.any() and ew is not None:
        optimism = float(np.clip(np.average(scores[pos_mask], weights=ew[pos_mask]), 0, 1))
    else:
        optimism = max(0.0, float(scores.mean()))
    anxiety = float(np.clip(0.5 * fear + 0.5 * (1 - calm), 0, 1))
    strongly_pos = (scores > 0.3).mean()
    happy = float(np.clip(0.6 * strongly_pos + 0.4 * keyword_boost(texts, HAPPY_KEYWORDS), 0, 1))

    daily.append({"date": str(date), "calm": round(calm, 4), "fear": round(fear, 4),
                  "optimism": round(optimism, 4), "anxiety": round(anxiety, 4), "happy": round(happy, 4)})

mood_df = pd.DataFrame(daily).sort_values("date").reset_index(drop=True)
mood_df.to_csv(DAILY_MOOD_CSV, index=False)
print(f"Mood computed for {len(mood_df)} days:\n")
mood_df

## Step 6: Fetch DJIA Price Data

In [ ]:
import yfinance as yf
from datetime import timedelta

end = datetime.now()
start = end - timedelta(days=LOOKBACK_DAYS)
print(f"Fetching {DJIA_TICKER} from {start.date()} to {end.date()}...")

ticker = yf.Ticker(DJIA_TICKER)
price_df = ticker.history(start=start, end=end, auto_adjust=True).reset_index()
price_df.columns = [c.lower().replace(" ", "_") for c in price_df.columns]

if "date" in price_df.columns:
    price_df["date"] = pd.to_datetime(price_df["date"]).dt.date
elif "datetime" in price_df.columns:
    price_df["date"] = pd.to_datetime(price_df["datetime"]).dt.date
    price_df.drop(columns=["datetime"], inplace=True)

keep = [c for c in ["date", "open", "high", "low", "close", "volume"] if c in price_df.columns]
price_df = price_df[keep]
price_df["daily_return"] = price_df["close"].pct_change()
price_df["direction"] = (price_df["daily_return"] > 0).astype(int)
price_df["volatility_5d"] = price_df["daily_return"].rolling(5).std()
price_df["ma_5"] = price_df["close"].rolling(5).mean()
price_df["ma_20"] = price_df["close"].rolling(20).mean()
price_df.dropna(inplace=True)
price_df["date"] = price_df["date"].astype(str)
price_df.to_csv(DJIA_PRICES_CSV, index=False)

print(f"Fetched {len(price_df)} trading days")
print(f"Latest: ${price_df.iloc[-1]['close']:,.2f} ({price_df.iloc[-1]['daily_return']:+.2%})")
price_df.tail()

## Step 7: Feature Engineering

In [ ]:
MOOD_COLS = ["calm", "fear", "optimism", "anxiety", "happy"]

mood = pd.read_csv(DAILY_MOOD_CSV)
prices = pd.read_csv(DJIA_PRICES_CSV)
mood["date"] = pd.to_datetime(mood["date"]).dt.strftime("%Y-%m-%d")
prices["date"] = pd.to_datetime(prices["date"]).dt.strftime("%Y-%m-%d")

feat_df = pd.merge(mood, prices, on="date", how="inner").sort_values("date").reset_index(drop=True)
n = len(feat_df)
print(f"Overlapping days: {n}")

if n >= 10:
    for col in MOOD_COLS:
        for lag in [1, 2, 3]: feat_df[f"{col}_lag{lag}"] = feat_df[col].shift(lag)
        feat_df[f"{col}_roll3"] = feat_df[col].rolling(3).mean()
        feat_df[f"{col}_mom"] = feat_df[col] - feat_df[col].shift(1)
elif n >= 4:
    for col in MOOD_COLS:
        feat_df[f"{col}_lag1"] = feat_df[col].shift(1)
        feat_df[f"{col}_mom"] = feat_df[col] - feat_df[col].shift(1)

feat_df["next_day_direction"] = feat_df["direction"].shift(-1)
feat_df.dropna(inplace=True)
if not feat_df.empty:
    feat_df["next_day_direction"] = feat_df["next_day_direction"].astype(int)
feat_df.to_csv(FEATURES_CSV, index=False)

EXCLUDE = {"date", "next_day_direction", "direction", "open", "high", "low", "close", "volume"}
feature_cols = [c for c in feat_df.columns if c not in EXCLUDE]
print(f"Feature matrix: {len(feat_df)} rows x {len(feature_cols)} features")
print(f"Features: {feature_cols}")
feat_df.head()

## Step 8: Train XGBoost Model

In [ ]:
import json
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

df = pd.read_csv(FEATURES_CSV)
X = df[feature_cols].astype(float)
y = df["next_day_direction"].astype(int)

print(f"Samples: {len(X)} | Features: {X.shape[1]}")
print(f"Class balance: {y.value_counts().to_dict()}")

if y.nunique() < 2:
    majority = int(y.mode()[0])
    print(f"\nOnly one class ({'UP' if majority else 'DOWN'}). Adding synthetic opposite to bootstrap.")
    print("Collect more data daily for a real model!")
    fake = X.iloc[[0]].copy()
    X = pd.concat([X, fake], ignore_index=True)
    y = pd.concat([y, pd.Series([1 - majority])], ignore_index=True)

if len(df) >= 5:
    split = max(2, int(len(df) * TRAIN_TEST_SPLIT))
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]
else:
    X_train, X_test = X, X
    y_train, y_test = y, y

model = xgb.XGBClassifier(**XGB_PARAMS)
model.fit(X_train, y_train, verbose=False)
model.save_model(str(MODEL_PATH))

y_pred = model.predict(X_test)
print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.2%}")
print(f"F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.2%}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.2%}")
print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.2%}")
print(f"\nModel saved to {MODEL_PATH}")

# Feature importance
imp = sorted(zip(feature_cols, model.feature_importances_), key=lambda x: x[1], reverse=True)
print("\nTop features:")
for feat, val in imp[:10]:
    print(f"  {feat:25s} {val:.4f} {'#' * int(val * 50)}")

## Step 9: Prediction

In [ ]:
model = xgb.XGBClassifier()
model.load_model(str(MODEL_PATH))

df = pd.read_csv(FEATURES_CSV)
latest = df.iloc[-1:]
X_latest = latest[feature_cols].astype(float)

pred = model.predict(X_latest)[0]
prob = model.predict_proba(X_latest)[0]
confidence = max(prob)
signal = "BULLISH" if pred == 1 else "BEARISH"

print("=" * 55)
print("  M O O D S I G N A L  --  PREDICTION")
print("=" * 55)
print(f"  Date:       {latest['date'].iloc[0]}")
print(f"  Signal:     {'>>>' if pred == 1 else 'vvv'}  {signal}")
print(f"  Confidence: {confidence:.1%}")
print()
print("  Mood Dimensions:")
for dim in ["calm", "fear", "optimism", "anxiety", "happy"]:
    if dim in latest.columns:
        val = float(latest[dim].iloc[0])
        bar = '#' * int(val * 20) + '.' * (20 - int(val * 20))
        print(f"    {dim.capitalize():10s} [{bar}] {val:.2%}")
print("=" * 55)

## Step 10: Download Your Data
Run this to download the spreadsheets to your local machine.

In [ ]:
from google.colab import files

for f in [RAW_POSTS_CSV, SCORED_POSTS_CSV, DAILY_MOOD_CSV, DJIA_PRICES_CSV, FEATURES_CSV]:
    if f.exists():
        files.download(str(f))
        print(f"Downloaded {f.name}")